# BREACHZERO - Person 3 (DistilBERT + eval)

Runtime -> Change runtime type -> **T4 GPU**. Then Runtime -> Run all.
Stops twice to ask for a token: Hugging Face (needs **write**), then GitHub at the end.

In [1]:
![ -d /content/BREACHZERO ] || git clone https://github.com/Priyanshu-2308/BREACHZERO.git /content/BREACHZERO

Cloning into '/content/BREACHZERO'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 82 (delta 28), reused 69 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 1.33 MiB | 4.24 MiB/s, done.
Resolving deltas: 100% (28/28), done.


In [2]:
%cd /content/BREACHZERO
!git pull --rebase origin main

/content/BREACHZERO
From https://github.com/Priyanshu-2308/BREACHZERO
 * branch            main       -> FETCH_HEAD
Already up to date.


In [3]:
# Colab's own torch/transformers are used as-is. Only sklearn gets a floor, because
# models/baseline.joblib was pickled with scikit-learn 1.9.1.
!pip install -q "scikit-learn>=1.9"

In [4]:
import joblib, sklearn, torch, transformers
print("sklearn", sklearn.__version__, "| transformers", transformers.__version__, "| torch", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - set the runtime to T4")

# fail fast: make sure the baseline loads before spending 15 minutes on training
pipe = joblib.load("models/baseline.joblib")
print("baseline check:", pipe.predict_proba(["ignore previous instructions"])[0][1].round(3))

sklearn 1.9.1 | transformers 5.22.1 | torch 2.14.1+cu130
gpu: Tesla T4
baseline check: 1.0


## Train both models

In [ ]:
import subprocess, sys

def run(*args):
    """Run a project script and STOP the notebook if it fails."""
    r = subprocess.run([sys.executable, *args])
    if r.returncode != 0:
        raise RuntimeError("FAILED: " + " ".join(args) + "  (scroll up for the real error)")

# 20-second preflight: catches import/API errors before the real 15-minute run
run("train.py", "--out", "/tmp/smoketest", "--epochs", "1", "--max_len", "64", "--limit", "32")
print("\npreflight OK")

In [5]:
run("train.py", "--out", "models/distilbert", "--epochs", "3")

model=distilbert-base-uncased  train=3258 (extra=0)  val=698  max_len=256
train label counts: {0: 2374, 1: 884}
config.json: 100% 483/483 [00:00<00:00, 1.56MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 273kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 2.99MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 25.7MB/s]

model.safetensors: downloading bytes:  83% 223M/268M [00:01<00:00, 320MB/s, 17.5MB/s  ]
model.safetensors: reconstructing file:  26% 68.6M/268M [00:01<00:03, 60.4MB/s]
model.safetensors: reconstructing file:  67% 181M/268M [00:01<00:00, 160MB/s, 6.65MB/s  ]
model.safetensors: downloading bytes: 100% 250M/250M [00:01<00:00, 152MB/s, 23.6MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:01<00:00, 162MB/s, 25.2MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 17109.13it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
voca

In [6]:
from config import MULTI_MODEL
run("train.py", "--model", MULTI_MODEL, "--out", "models/distilbert_multilingual", "--epochs", "3")

model=distilbert-base-multilingual-cased  train=3258 (extra=0)  val=698  max_len=256
train label counts: {0: 2374, 1: 884}
config.json: 100% 466/466 [00:00<00:00, 1.75MB/s]
tokenizer_config.json: 100% 49.0/49.0 [00:00<00:00, 200kB/s]
vocab.txt: 100% 996k/996k [00:00<00:00, 8.04MB/s]
tokenizer.json: 100% 1.96M/1.96M [00:00<00:00, 39.2MB/s]

model.safetensors: downloading bytes:  59% 319M/542M [00:01<00:00, 228MB/s, 27.8MB/s  ]
model.safetensors: downloading bytes:  94% 508M/542M [00:02<00:00, 336MB/s, 41.9MB/s  ]
model.safetensors: downloading bytes: 100% 511M/511M [00:02<00:00, 195MB/s, 45.5MB/s  ]
model.safetensors: reconstructing file: 100% 542M/542M [00:02<00:00, 207MB/s, 49.3MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 20220.33it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-multilingual-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weigh

## Evaluate

In [7]:
run("run_eval.py")


evaluating baseline ...
SKIP distilbert: models/distilbert not found (train it with train.py, or download it as described in MODELS.md)
SKIP distilbert_multilingual: models/distilbert_multilingual not found (train it with train.py, or download it as described in MODELS.md)

threshold = 0.5
model                     test_set            recall        FPR       f1     n
baseline                  test                0.9632     0.0098   0.9683   699
baseline                  ood_jailbreak       0.8786     0.1724   0.8582  1289
baseline                  leetspeak           0.4947          -   0.6620   190
baseline                  cyrillic            0.0000          -   0.0000   190
baseline                  base64              0.0000          -   0.0000   190
baseline                  wrapped             0.8895          -   0.9415   190
baseline                  hindi               0.4368          -   0.6081   190

Saved 7 rows to results/main_results.csv
run_eval.py finished.


In [8]:
import pandas as pd
pd.read_csv("results/main_results.csv")

,model,test_set,recall,false_positive_rate,f1,n
0,baseline,test,0.963158,0.009823,0.968254,699
1,baseline,ood_jailbreak,0.878648,0.172414,0.858215,1289
2,baseline,leetspeak,0.494737,NaN,0.661972,190
3,baseline,cyrillic,0.000000,NaN,0.000000,190
4,baseline,base64,0.000000,NaN,0.000000,190
5,baseline,wrapped,0.889474,NaN,0.941504,190
6,baseline,hindi,0.436842,NaN,0.608059,190


## Push models to the Hub, fill in MODELS.md

In [9]:
from huggingface_hub import notebook_login
notebook_login()

In [10]:
import pathlib
from huggingface_hub import HfApi, whoami

api = HfApi()
user = whoami()["name"]
print("logged in as", user)

repos = {
    "models/distilbert": "breachzero-distilbert",
    "models/distilbert_multilingual": "breachzero-distilbert-multilingual",
}
for folder, name in repos.items():
    repo_id = user + "/" + name
    api.create_repo(repo_id, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=folder, repo_id=repo_id)
    print("pushed", repo_id)

md_path = pathlib.Path("MODELS.md")
md_path.write_text(md_path.read_text().replace("<HF_USERNAME>", user))
print(md_path.read_text())

logged in as sarthak776


ValueError: Provided path: '/content/BREACHZERO/models/distilbert' is not a directory

## Commit results

Should list only `MODELS.md` and `results/main_results.csv`.

In [ ]:
!git status --short

In [ ]:
from getpass import getpass

token = getpass("GitHub token: ")

!git config user.name "sarthak0707git"
!git config user.email "sarthak776@gmail.com"
!git add MODELS.md results/main_results.csv
!git commit -m "P3: main results table and Hugging Face model links"
!git pull --rebase origin main
!git push https://{token}@github.com/Priyanshu-2308/BREACHZERO.git main